# Task 5: BERT
BERT is a language model that was already trained on a very large amount of text.
We fine-tune it: we train it a little more on our messages, so it learns to find spam.
We use Google Colab because the model needs a GPU.

In [1]:
import torch
import transformers
import pandas as pd
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('GPU:', torch.cuda.is_available())
train = pd.read_csv('train_prep_v2.csv')
test = pd.read_csv('test_prep_v2.csv')
print(train.shape, test.shape)
print(train['label'].value_counts().to_dict())

torch: 2.14.1+cu130
transformers: 5.22.1
GPU: True
(4102, 9) (1026, 9)
{'ham': 3601, 'spam': 501}


## 1. Tokenize the messages
BERT does not use our cleaned text. It reads the raw message.
It has its own tokenizer: it cuts a word into small pieces when it does not know the word.
We use the raw `text` column, and the same messages as before.
We also look at the length of the messages, to choose a maximum length.

In [2]:
from transformers import AutoTokenizer
MODEL = 'bert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL)
spam = train[train['label'] == 'spam'].iloc[0]['text']
print(spam)
print(tok.tokenize(spam))
enc = tok(list(train['text']))
lengths = pd.Series([len(x) for x in enc['input_ids']])
print(lengths.describe().round(1))
for n in [64, 96, 128, 160]:
    print(n, 'tokens covers', round(100 * (lengths <= n).mean(), 1), '% of messages')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Please call Amanda with regard to renewing or upgrading your current T-Mobile handset free of charge. Offer ends today. Tel 0845 021 3680 subject to T's and C's
['please', 'call', 'amanda', 'with', 'regard', 'to', 'renew', '##ing', 'or', 'upgrading', 'your', 'current', 't', '-', 'mobile', 'hands', '##et', 'free', 'of', 'charge', '.', 'offer', 'ends', 'today', '.', 'tel', '08', '##45', '02', '##1', '36', '##80', 'subject', 'to', 't', "'", 's', 'and', 'c', "'", 's']
count    4102.0
mean       24.9
std        17.0
min         4.0
25%        12.0
50%        19.0
75%        34.0
max       238.0
dtype: float64
64 tokens covers 98.4 % of messages
96 tokens covers 99.5 % of messages
128 tokens covers 99.9 % of messages
160 tokens covers 99.9 % of messages


## 2. Prepare the data
We use the same split as the LSTM notebook:
a part of the train data for checking (validation), with the same random seed.
So the BERT model and the LSTM model learn from the same messages.
We choose 64 tokens as the maximum length. It covers 98.4% of the train messages.
Short messages get empty places at the end, and long messages are cut.

In [3]:
from sklearn.model_selection import train_test_split
MAX_LEN = 64
y_train = (train['label'] == 'spam').astype(int).values
y_test = (test['label'] == 'spam').astype(int).values
txt_fit, txt_val, y_fit, y_val = train_test_split(
    list(train['text']), y_train, test_size=0.2,
    random_state=42, stratify=y_train)
print('Fit  :', len(txt_fit), '| spam:', y_fit.sum())
print('Valid:', len(txt_val), '| spam:', y_val.sum())

def encode(texts):
    return tok(texts, max_length=MAX_LEN, truncation=True,
               padding='max_length', return_tensors='pt')

enc_fit = encode(txt_fit)
enc_val = encode(txt_val)
enc_test = encode(list(test['text']))
print(enc_fit['input_ids'].shape, enc_val['input_ids'].shape, enc_test['input_ids'].shape)
print(list(enc_fit.keys()))

Fit  : 3281 | spam: 401
Valid: 821 | spam: 100
torch.Size([3281, 64]) torch.Size([821, 64]) torch.Size([1026, 64])
['input_ids', 'token_type_ids', 'attention_mask']


## 3. Load the BERT model
We load `bert-base-uncased`. It has 12 layers and it was trained on a very large text.
We add one small layer on top of it. This layer gives two numbers: ham and spam.
This new layer starts with random numbers, and the training will teach it.
We give the messages to the model in small groups of 32 (batches).
We fix the random seed, as we did for the LSTM.

In [4]:
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForSequenceClassification

torch.manual_seed(42)
np.random.seed(42)
torch.cuda.manual_seed_all(42)
device = torch.device('cuda')

def make_loader(enc, y, shuffle):
    data = TensorDataset(enc['input_ids'], enc['attention_mask'],
                         torch.tensor(y))
    return DataLoader(data, batch_size=32, shuffle=shuffle)

loader_fit = make_loader(enc_fit, y_fit, True)
loader_val = make_loader(enc_val, y_val, False)
loader_test = make_loader(enc_test, y_test, False)
print('Batches:', len(loader_fit), len(loader_val), len(loader_test))

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model.to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))

Batches: 103 26 33


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 109483778


## 4. Training function
We write one function that goes through all the messages one time (one epoch).
- When it trains, the model learns and changes its numbers.
- When it only checks, the model does not change.
It gives the loss and the accuracy.
We use the AdamW optimizer with a small learning rate (0.00002),
because BERT is already trained and we only need to change it a little.
Before the training, we check the model on the validation part.
It must be bad, because the new layer is still random.

In [5]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=2e-5)

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0, 0, 0
    for ids, mask, labels in loader:
        ids = ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device)
        with torch.set_grad_enabled(train):
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
        if train:
            optimizer.zero_grad()
            out.loss.backward()
            optimizer.step()
        total_loss += out.loss.item() * len(labels)
        correct += (out.logits.argmax(dim=1) == labels).sum().item()
        n += len(labels)
    return total_loss / n, correct / n

loss, acc = run_epoch(loader_val, False)
print('Before training: val_loss', round(loss, 4), '| val_accuracy', round(acc, 4))

Before training: val_loss 0.7321 | val_accuracy 0.2132


## 5. Train the model
We train the model with the same rule as the LSTM.
After every epoch we check the validation part.
If the validation loss does not get better for 3 epochs, we stop.
Then we go back to the weights of the best epoch.
We save these weights in memory every time the validation loss is the best.

In [6]:
import copy
history = []
best_loss = 999
best_epoch = 0
best_state = None
wait = 0
for epoch in range(1, 11):
    tr_loss, tr_acc = run_epoch(loader_fit, True)
    va_loss, va_acc = run_epoch(loader_val, False)
    history.append([epoch, tr_acc, tr_loss, va_acc, va_loss])
    print('Epoch', epoch, '| loss', round(tr_loss, 4), '| acc', round(tr_acc, 4),
          '| val_loss', round(va_loss, 4), '| val_acc', round(va_acc, 4))
    if va_loss < best_loss:
        best_loss = va_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait == 3:
            break
model.load_state_dict(best_state)
print('Stopped after epoch', epoch, '| best epoch', best_epoch)

Epoch 1 | loss 0.1439 | acc 0.9457 | val_loss 0.0446 | val_acc 0.9903
Epoch 2 | loss 0.0265 | acc 0.9945 | val_loss 0.0437 | val_acc 0.9903
Epoch 3 | loss 0.0142 | acc 0.9963 | val_loss 0.0506 | val_acc 0.989
Epoch 4 | loss 0.0064 | acc 0.9982 | val_loss 0.0467 | val_acc 0.9878
Epoch 5 | loss 0.003 | acc 0.9991 | val_loss 0.0631 | val_acc 0.989
Stopped after epoch 5 | best epoch 2


## 6. Test the model
Now we use the test data for the first time.
The model gives a probability of spam for every message.
If the number is 0.5 or more, we say spam.
This is the same rule as the LSTM.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix

model.eval()
probs = []
with torch.no_grad():
    for ids, mask, labels in loader_test:
        out = model(input_ids=ids.to(device), attention_mask=mask.to(device))
        p = torch.softmax(out.logits, dim=1)[:, 1]
        probs.extend(p.cpu().numpy())
prob = np.array(probs)
pred = (prob >= 0.5).astype(int)
print('Accuracy :', round(100 * accuracy_score(y_test, pred), 1))
print('Precision:', round(100 * precision_score(y_test, pred), 1))
print('Recall   :', round(100 * recall_score(y_test, pred), 1))
print('F1       :', round(100 * f1_score(y_test, pred), 1))
cm = confusion_matrix(y_test, pred)
print(pd.DataFrame(cm, index=['true ham', 'true spam'],
                   columns=['said ham', 'said spam']))

Accuracy : 99.0
Precision: 98.3
Recall   : 93.6
F1       : 95.9
           said ham  said spam
true ham        899          2
true spam         8        117


## 7. Save the predictions and the training history
We save what the model said about every test message,
with the same columns as the other models (label, text, pred, prob_spam).
We also save the training history, so Abdullah can draw the learning curves.

In [8]:
out = test[['label', 'text']].copy()
out['pred'] = np.where(pred == 1, 'spam', 'ham')
out['prob_spam'] = prob.round(4)
out.to_csv('pred_bert.csv', index=False)
print(out['pred'].value_counts().to_dict())

hist = pd.DataFrame(history, columns=['epoch', 'accuracy', 'loss',
                                      'val_accuracy', 'val_loss'])
hist = hist.set_index('epoch').round(4)
hist.to_csv('bert_history.csv')
print(hist)

{'ham': 907, 'spam': 119}
       accuracy    loss  val_accuracy  val_loss
epoch                                          
1        0.9457  0.1439        0.9903    0.0446
2        0.9945  0.0265        0.9903    0.0437
3        0.9963  0.0142        0.9890    0.0506
4        0.9982  0.0064        0.9878    0.0467
5        0.9991  0.0030        0.9890    0.0631
